# 🚀 Demostración Interactiva del Pipeline de 10 Etapas (Streaming de E-commerce)
Este cuaderno interactivo recorre paso a paso el flujo de procesamiento de datos en tiempo real, desde la ingesta en búfer hasta las estructuras de búsqueda avanzada, indexación, grafos y analíticas finales.

In [42]:
import os
import sys
import pandas as pd

# Ajuste automático de ruta si el notebook está en 'notebooks/'
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"📁 Directorio de trabajo: {os.getcwd()}")

# Cargar dataset base de 1K para la demostración interactiva
target_file = 'data/ecommerce_1K.parquet'
if os.path.exists(target_file):
    df_stream = pd.read_parquet(target_file)
    print(f"✅ Dataset cargado exitosamente. Total registros: {len(df_stream)}")
else:
    print(f"⚠️ No se encontró {target_file}. Asegúrate de haber generado los parquets.")

📁 Directorio de trabajo: c:\Users\Jesus\proyecto-eda
✅ Dataset cargado exitosamente. Total registros: 1000


---
### 📥 ETAPA 0: Ingesta y Buffer (Stream de Entrada - FIFO)
Simulamos la llegada de eventos en streaming estructurándolos en una cola FIFO con capacidad limitada.

In [43]:
from src.etapa_0_ingesta_buffer import IngestaBufferFIFO

buffer = IngestaBufferFIFO(capacity=5000)
for _, row in df_stream.head(500).iterrows():
    buffer.enqueue(row.to_dict())

print(f"📦 Total de eventos en el Buffer FIFO: {buffer.size()}")

# Extraemos un lote para procesarlo en el pipeline
lote_crudo = [buffer.dequeue() for _ in range(100)]
print(f"🔍 Lote crudo extraído para procesamiento: {len(lote_crudo)} eventos.")

📦 Total de eventos en el Buffer FIFO: 500
🔍 Lote crudo extraído para procesamiento: 100 eventos.


---
### 🧹 ETAPA 1: Preprocesamiento por Lotes (Timsort + Búsqueda Binaria)
Limpieza de nulos, eliminación de duplicados y ordenamiento cronológico del lote.

In [44]:
from src.etapa_1_preprocesamiento_lotes import PreprocesamientoLotes

preproc = PreprocesamientoLotes()
lote_dedup = preproc.deduplicate_batch(lote_crudo)
lote_ordenado = preproc.sort_batch(lote_dedup, key='event_time')

print(f"✨ Lote limpio, deduplicado y ordenado: {len(lote_ordenado)} eventos listos.")

✨ Lote limpio, deduplicado y ordenado: 100 eventos listos.


---
### ⚡ ETAPA 2: Cola de Prioridades (MinHeap)
Organización de los eventos según su criticidad de negocio (ej. transacciones de compra prioritarias).

In [45]:
from src.etapa_2_cola_prioridades import ColaPrioridadesHeap

cola_prioridad = ColaPrioridadesHeap()
for ev in lote_ordenado:
    ev['priority'] = 1 if ev.get('event_type') == 'purchase' else 3
    cola_prioridad.push(ev)

print(f"📊 Elementos en Cola de Prioridades (MinHeap): {cola_prioridad.size()}")
lote_priorizado = [cola_prioridad.pop() for _ in range(len(lote_ordenado))]

📊 Elementos en Cola de Prioridades (MinHeap): 100


---
### 💾 ETAPA 3: Almacenamiento, Lookup y Caché LRU
Evaluación del rendimiento al consultar eventos mediante una política de caché rápida (LRU).

In [46]:
from src.etapa_3_almacenamiento_lookup_cache import AlmacenamientoLookupCache

storage_cache = AlmacenamientoLookupCache(capacity=20)
for ev in lote_priorizado:
    storage_cache.put_storage(ev['event_id'], ev)

sample_id = lote_priorizado[0]['event_id']
_, status1 = storage_cache.get_from_cache_or_storage(sample_id) # 1er acceso (MISS)
_, status2 = storage_cache.get_from_cache_or_storage(sample_id) # 2do acceso (HIT)

print(f"🔎 Consulta de Evento ID {sample_id}:")
print(f"   - 1er Acceso (Caché Miss): {status1}")
print(f"   - 2do Acceso (Caché Hit): {status2}")

🔎 Consulta de Evento ID a55998e3:
   - 1er Acceso (Caché Miss): CACHE_MISS
   - 2do Acceso (Caché Hit): CACHE_HIT


---
### 🗂️ ETAPA 4: Indexación Principal (Árbol B+)
Indexación de claves numéricas en disco para búsquedas estructuradas eficientes.

In [47]:
from src.etapa_4_indexacion_principal import IndexacionPrincipalBPlusTree

b_plus = IndexacionPrincipalBPlusTree(t=2)
for ev in lote_priorizado[:30]:
    numeric_key = abs(hash(ev['event_id'])) % 100000
    b_plus.insert(numeric_key, {'event_type': ev['event_type']})

print("✅ Índices de eventos estructurados correctamente en el Árbol B+ (t=2).")

✅ Índices de eventos estructurados correctamente en el Árbol B+ (t=2).


---
### 📐 ETAPA 5: Índice Multidimensional y KNN Exacto (KD-Tree)
Búsqueda de vecinos más cercanos basada en atributos numéricos del usuario (precio, IDs).

In [48]:
from src.etapa_5_indice_multidimensional import IndiceMultidimensionalKDTree

points = [(float(ev['price']), int(ev['user_id'])) for ev in lote_priorizado if ev.get('price') is not None]
data_ids = [ev['event_id'] for ev in lote_priorizado if ev.get('price')]

if points:
    kd_tree = IndiceMultidimensionalKDTree(points, data_ids)
    query_pt = points[0]
    knn_res = kd_tree.knn_exact_baseline(query_pt, points, data_ids, k_neighbors=2)
    print(f"📍 Punto de consulta (Precio, User ID): {query_pt}")
    print(f"✅ Vecinos más cercanos (KNN Baseline): {knn_res}")

📍 Punto de consulta (Precio, User ID): (489.07, 520088904)
✅ Vecinos más cercanos (KNN Baseline): ['a55998e3', '3f12b6d5']


---
### 🛡️ ETAPA 6: Filtrado Probabilístico (Bloom Filter y Count-Min Sketch)
Consulta rápida de membresía de productos y estimación de frecuencias en streaming.

In [49]:
from src.etapa_6_filtrado_probabilistico import BloomFilter, CountMinSketch

bloom = BloomFilter(capacity=1000)
cms = CountMinSketch()

for ev in lote_priorizado:
    prod_id = str(ev.get('product_id'))
    bloom.add(prod_id)
    cms.add(prod_id)

test_prod = str(lote_priorizado[0].get('product_id'))
print(f"🔍 ¿Producto {test_prod} presente en Bloom Filter?: {test_prod in bloom}")
print(f"📈 Frecuencia estimada (Count-Min Sketch) para {test_prod}: {cms.estimate_frequency(test_prod)}")

🔍 ¿Producto 1003461 presente en Bloom Filter?: True
📈 Frecuencia estimada (Count-Min Sketch) para 1003461: 1


---
### 🌐 ETAPA 7: Similitud Aproximada (MinHash LSH y HNSW)
Comparación de canastas de usuarios y búsqueda vectorial de alta velocidad (ANN).

In [50]:
from src.etapa_7_similitud_aproximada import HNSWApproximateSearch, MinHashLSHSimulator

lsh_sim = MinHashLSHSimulator(num_perm=32)
user_baskets = {}
for ev in lote_priorizado:
    uid = ev['user_id']
    user_baskets.setdefault(uid, set()).add(ev['product_id'])

u_ids = list(user_baskets.keys())
if len(u_ids) >= 2:
    sig1 = lsh_sim.compute_signature(user_baskets[u_ids[0]])
    sig2 = lsh_sim.compute_signature(user_baskets[u_ids[1]])
    jaccard_est = lsh_sim.jaccard_similarity_estimate(sig1, sig2)
    print(f"🔗 Similitud Jaccard estimada entre canastas de usuarios: {jaccard_est:.2f}")

hnsw = HNSWApproximateSearch()
for i, ev in enumerate(lote_priorizado[:15]):
    vec = [float(ev.get('price', 0)) / 100.0, float(i) / 15.0]
    hnsw.add_vector(ev['event_id'], vec)

if lote_priorizado:
    query_vec = [float(lote_priorizado[0].get('price', 0)) / 100.0, 0.0]
    neighbors_hnsw = hnsw.search_knn(query_vec, k=1)
    print(f"🎯 Búsqueda Vectorial ANN (HNSW) más cercana: {neighbors_hnsw}")

🔗 Similitud Jaccard estimada entre canastas de usuarios: 0.00
🎯 Búsqueda Vectorial ANN (HNSW) más cercana: ['a55998e3']


---
### 🕸️ ETAPA 8: Grafo de Dominio (Redes, PageRank y Comunidades)
Modelado de interacciones Usuario-Producto, cálculo de PageRank e identificación de comunidades.

In [51]:
from src.etapa_8_grafo_dominio import GrafoDominioRed

grafo = GrafoDominioRed()
for ev in lote_priorizado:
    u_node = f"user_{ev['user_id']}"
    p_node = f"prod_{ev['product_id']}"
    grafo.add_edge(u_node, p_node, weight=1.0)

pagerank_res = grafo.compute_pagerank()
top_nodes = sorted(pagerank_res.items(), key=lambda x: x[1], reverse=True)[:3]
print(f"👑 Top 3 Nodos más influyentes (PageRank): {top_nodes}")

communities = grafo.detect_communities()
print(f"👥 Comunidades transaccionales detectadas: {len(communities)} clústeres")

👑 Top 3 Nodos más influyentes (PageRank): [('prod_12708937', 0.0111051855869166), ('prod_5000088', 0.007608802204875131), ('prod_1004775', 0.007608802204875131)]
👥 Comunidades transaccionales detectadas: 78 clústeres


---
### 📊 ETAPA 9: Servicio y Visualización (Top-K Analytics & Trie de Autocompletado)
Analíticas de popularidad en tiempo real y sugerencias de búsqueda con árboles Trie.

In [52]:
from src.etapa_9_servicio_visualizacion import TopKAnalytics, TrieAutocompletado

trie = TrieAutocompletado()
top_k = TopKAnalytics(k=3)

for ev in lote_priorizado:
    prod_id_str = str(ev['product_id'])
    top_k.add_item(prod_id_str)
    trie.insert(f"producto {prod_id_str}")

print(f"🏆 Top 3 Productos más populares (Heap): {top_k.get_top_k()}")
print(f"🔤 Sugerencias de autocompletado para 'pro': {trie.search_prefix('pro')[:3]}")
print("\n🎉 ¡DEMOSTRACIÓN INTERACTIVA DEL PIPELINE COMPLETADA CON ÉXITO! 🎉")


🏆 Top 3 Productos más populares (Heap): [('12708937', 2), ('27400066', 2), ('16700826', 2)]
🔤 Sugerencias de autocompletado para 'pro': ['producto 1003461', 'producto 1003317', 'producto 1003316']

🎉 ¡DEMOSTRACIÓN INTERACTIVA DEL PIPELINE COMPLETADA CON ÉXITO! 🎉
